# Module 4 — Bioactivity Heatmap

This notebook creates a heatmap showing which endophyte-derived compounds have which bioactivities, based on the literature review.

**Two versions provided:**
- **Python version** (run directly in Colab below)
- The R script `04_bioactivity_heatmap.R` in your repo is for RStudio if you ever install it

**This notebook is independent — no other notebook needs to run first.**

In [ ]:
# ── CELL 1: Imports ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

os.makedirs("figures", exist_ok=True)
print("✓ Ready")

In [ ]:
# ── CELL 2: Define bioactivity matrix ──
# 1 = activity reported in literature, 0 = not reported
# Source: Banerjee (2019), NTCC Review Article, Amity University Kolkata

compounds = [
    "Taxol", "Camptothecin", "Podophyllotoxin",
    "Ergoflavin", "Torreyanic acid", "Secalonic acid D",
    "Griseofulvin", "Coronamycin", "Hypericin",
    "Kakadumycin A", "Brefeldin A", "Helvolic acid",
    "Emodin", "Pestacin", "Isopestacin",
    "Huperzine A", "Subglutinol A"
]

bioactivities = [
    "Anticancer", "Antimicrobial", "Antifungal",
    "Antioxidant", "Antiviral", "Immunosuppressive", "Neuroprotective"
]

#                     Anti  Antim Antif Antio Antiv Immun Neuro
data = np.array([
    [1, 0, 0, 0, 0, 0, 0],  # Taxol
    [1, 0, 1, 0, 0, 0, 0],  # Camptothecin
    [1, 0, 0, 0, 1, 0, 0],  # Podophyllotoxin
    [1, 0, 0, 0, 0, 0, 0],  # Ergoflavin
    [1, 0, 0, 0, 0, 0, 0],  # Torreyanic acid
    [1, 0, 0, 0, 0, 0, 0],  # Secalonic acid D
    [0, 1, 1, 0, 0, 0, 0],  # Griseofulvin
    [0, 1, 1, 0, 0, 0, 0],  # Coronamycin
    [0, 1, 0, 0, 1, 0, 0],  # Hypericin
    [0, 1, 0, 0, 0, 0, 0],  # Kakadumycin A
    [0, 1, 1, 0, 0, 0, 0],  # Brefeldin A
    [0, 1, 0, 0, 0, 0, 0],  # Helvolic acid
    [0, 1, 1, 0, 0, 0, 0],  # Emodin
    [0, 0, 1, 1, 0, 0, 0],  # Pestacin
    [0, 0, 1, 1, 0, 0, 0],  # Isopestacin
    [0, 0, 0, 0, 0, 0, 1],  # Huperzine A
    [0, 0, 0, 0, 0, 1, 0],  # Subglutinol A
])

# Sort compounds by number of bioactivities (most versatile first)
row_sums   = data.sum(axis=1)
sort_order = np.argsort(row_sums)[::-1]
data       = data[sort_order]
compounds  = [compounds[i] for i in sort_order]

df_matrix = pd.DataFrame(data, index=compounds, columns=bioactivities)
print("✓ Matrix built")
df_matrix

In [ ]:
# ── CELL 3: Plot heatmap ──
fig, ax = plt.subplots(figsize=(9, 7))

cmap = mcolors.ListedColormap(["#F5F5F5", "#1565C0"])

im = ax.imshow(data, cmap=cmap, aspect="auto", vmin=0, vmax=1)

# Gridlines
ax.set_xticks(np.arange(len(bioactivities)))
ax.set_yticks(np.arange(len(compounds)))
ax.set_xticklabels(bioactivities, rotation=35, ha="right", fontsize=10)
ax.set_yticklabels(compounds, fontsize=10)

ax.set_xticks(np.arange(-0.5, len(bioactivities), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(compounds), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

# Annotate cells
for i in range(len(compounds)):
    for j in range(len(bioactivities)):
        val = data[i, j]
        text_color = "white" if val == 1 else "#BDBDBD"
        ax.text(j, i, "●" if val == 1 else "",
                ha="center", va="center",
                color=text_color, fontsize=14)

# Colorbar legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#1565C0", label="Reported in literature"),
    Patch(facecolor="#F5F5F5", edgecolor="#DDD", label="Not reported")
]
ax.legend(handles=legend_elements, loc="lower right",
          bbox_to_anchor=(1.0, -0.22), fontsize=9, frameon=False)

ax.set_title("Bioactivity Profile of Endophyte-Derived Compounds",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Bioactivity", fontsize=11, labelpad=10)

plt.tight_layout()
plt.savefig("figures/bioactivity_heatmap.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Figure saved to figures/bioactivity_heatmap.png")

In [ ]:
# ── CELL 4: Summary statistics ──
print("=== Bioactivity Summary ===")
print(f"Total compounds analysed: {len(compounds)}")
print()

print("Compounds by number of bioactivities:")
for compound, row_sum in zip(compounds, data.sum(axis=1)):
    bar = "█" * int(row_sum)
    print(f"  {compound:<25} {bar} ({int(row_sum)})")

print()
print("Most represented bioactivity categories:")
for bioact, col_sum in zip(bioactivities, data.sum(axis=0)):
    bar = "█" * int(col_sum)
    print(f"  {bioact:<20} {bar} ({int(col_sum)} compounds)")

In [ ]:
# ── CELL 5: Download figure ──
from google.colab import files
files.download("figures/bioactivity_heatmap.png")
print("✓ Downloaded! Upload to your GitHub repo under figures/")